# Hebrew Acronym Disambiguation: Evaluation Protocol

This computational appendix evaluates Hebrew acronym disambiguation under two task formulations: **free generation**, without candidate expansions or reference answers, and **candidate selection**, with a fixed candidate inventory and recorded candidate order. Each language model is evaluated on the same items in both tasks.

The technical pilot uses the first **10 of 289 development items** in `data/splits/dev_items.csv`. The test set contains **395 items**: 267 from Knesset text, 41 from Wikipedia, and 87 AI-authored items. It is therefore not a natural-text-only benchmark. Source provenance is documented in `data/README.md` and `data/mined/DATASET_CARD.md` in the source archive. The separate 62-item encoder development cohort is not the pilot cohort used here.

The protocol separates environment preparation, technical pilot inspection, and test evaluation. Software checks establish implementation properties; they do not establish live service availability, successful collection, or model performance. Results are supported by the identified run artifacts produced during execution. The operational reference is `docs/reproducible_evaluation.md` in the source archive.


## 1. Source and Environment Reproducibility

`CODE_REVISION` identifies an exact 40-character Git commit. In archive mode, upload `evaluation-release.tar` through the Colab Files panel and set `ARCHIVE_PATH`; `ARCHIVE_SHA256` must match the independently supplied release checksum. The installation checks both the archive checksum and its embedded commit before installing the package. Archive mode does not require publishing the commit. The alternative Git mode requires that the specified commit be accessible remotely.

The installation verifies the project's pinned dependencies and reports conflicts among other preinstalled Colab packages separately. Restart the runtime and repeat setup after changing the source revision or when Colab requests a restart for dependencies already loaded in memory. Subsequent cells use the installed package.


In [1]:
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import tarfile
import tempfile
import time
import urllib.request

CODE_REVISION = "64f47b76317d61868765e4ad05ecf12ddc6f4379" # Exact 40-character source commit; branch names are not valid identities.
SOURCE_MODE = "archive"  # "archive" or "git"
ARCHIVE_PATH = Path("/content/evaluation-release.tar")
ARCHIVE_SHA256 = "7f4b426451e1c38777b3f169db76b4dd56bc65aae4f67d391e72914414dbf7d2" # Independently supplied release checksum.
REPOSITORY_URL = "https://github.com/BenCarmel123/hebrew-acronym-disambiguation.git"
if not re.fullmatch(r"[0-9a-f]{40}", CODE_REVISION):
    raise ValueError("Set the reviewed full commit SHA before installation")
REPO_DIR = Path("/content/hebrew-eval-" + CODE_REVISION)
REPO_DIR.mkdir(parents=True, exist_ok=True)
if SOURCE_MODE == "archive":
    if not re.fullmatch(r"[0-9a-f]{64}", ARCHIVE_SHA256):
        raise ValueError("Set the release archive SHA-256")
    archive_bytes = ARCHIVE_PATH.read_bytes()
    if hashlib.sha256(archive_bytes).hexdigest() != ARCHIVE_SHA256:
        raise ValueError("Uploaded archive differs from the reviewed release")
    archive_revision = subprocess.check_output(
        ["git", "get-tar-commit-id"], input=archive_bytes).decode().strip()
    if archive_revision != CODE_REVISION:
        raise ValueError("Archive commit differs from CODE_REVISION")
    with tarfile.open(ARCHIVE_PATH, "r:") as archive:
        archive.extractall(REPO_DIR, filter="data")
elif SOURCE_MODE == "git":
    subprocess.run(["git", "init", str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1",
                    REPOSITORY_URL, CODE_REVISION], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
    actual = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
    if actual != CODE_REVISION:
        raise ValueError("Fetched source differs from CODE_REVISION")
else:
    raise ValueError("Choose archive or git")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(REPO_DIR)], check=True)
# Regular installation is immediately importable in this running notebook kernel.
# Check this project's pins; Colab can contain unrelated preinstalled conflicts.
for line in (REPO_DIR / "requirements.txt").read_text().splitlines():
    requirement = line.strip()
    if requirement and not requirement.startswith("#"):
        package_name, required_version = requirement.split("==", 1)
        if importlib.metadata.version(package_name) != required_version:
            raise RuntimeError(f"Required package version mismatch: {package_name}")
pip_report = subprocess.run([sys.executable, "-m", "pip", "check"],
                            capture_output=True, text=True)
if pip_report.returncode:
    print("Colab environment dependency report (inspect separately from project pins):")
    print(pip_report.stdout or pip_report.stderr)
print("Installed source commit:", CODE_REVISION)

Colab environment dependency report (inspect separately from project pins):
ipython 7.34.0 requires jedi, which is not installed.
google-colab 1.0.0 has requirement requests==2.32.4, but you have requests 2.32.5.
numba 0.61.2 has requirement numpy<2.3,>=1.24, but you have numpy 2.5.3.
torchvision 0.26.0+cu130 has requirement torch==2.11.0, but you have torch 2.8.0.
diffusers 0.40.0 has requirement huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2.
gradio 6.29.0 has requirement huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2.

Installed source commit: 64f47b76317d61868765e4ad05ecf12ddc6f4379


## 2. Persistent Storage, Credentials, and Expenditure

`RUN_NAME` identifies the persistent Drive directory. Retain this name and the same settings when resuming after a disconnection or selecting additional systems. A changed code revision or configuration requires a new identified session; preserve previous directories and carry their expenditure into the new session exactly once.

Set `PRIOR_SPEND_ILS` to earlier API expenditure outside this session, including failed requests and conservative allowances for uncertain charges. Its initial value, `None`, requires explicit reconciliation before execution. The combined expenditure ceiling is **100 ILS** across systems, pilots, test requests, and retries. The conversion allowance of 4 ILS/USD includes a fee and tax margin; it is not a quoted exchange rate. Changing system selection does not reset expenditure.

`SELECTED_SYSTEMS` determines which of the six systems are attempted. Store `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GEMINI_API_KEY`, and `XAI_API_KEY` in **Colab Secrets** and enable access for this notebook. Credentials must not appear in notebook cells or outputs. Missing credentials are reported per system and do not prevent other systems from proceeding. Use one writer per run directory; mounted Drive storage does not guarantee durability during a storage failure. Paid runtime purchases and automatic credit replenishment are outside this protocol.


In [3]:
from google.colab import drive, userdata
import importlib.metadata
import platform
import torch
from hebrew_acronyms.staged_evaluation import (
    SYSTEM_NAMES, prepare_session, run_system_pilot, review_system_pilot,
    run_system_full, session_summary,
)
from hebrew_acronyms.test_evaluation import summarize_evaluation
from hebrew_acronyms.models.qwen.eval import inspect_ollama
from hebrew_acronyms.models.openai.eval import inspect_openai_model
from hebrew_acronyms.models.anthropic.eval import inspect_anthropic_model
from hebrew_acronyms.models.xai.eval import inspect_xai_model
from hebrew_acronyms.models.common.http import request_json

RUN_NAME = "evaluation-xai-qwen14-20261009-64f47b7"
SELECTED_SYSTEMS = ["qwen14", "xai"]
PRIOR_SPEND_ILS = 4.6070994 # Required: reconciled prior expenditure and uncertain-cost allowances.
ILS_PER_USD_ALLOWANCE = 4.0  # Conversion allowance including fees and taxes; not a quoted exchange rate.
if not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_NAME):
    raise ValueError("Use a simple stable run name")
if not SELECTED_SYSTEMS or len(set(SELECTED_SYSTEMS)) != len(SELECTED_SYSTEMS) or not set(SELECTED_SYSTEMS) <= set(SYSTEM_NAMES):
    raise ValueError("Select one or more of the approved systems, without duplicates")
if PRIOR_SPEND_ILS is None:
    raise ValueError("Account for earlier attempts in PRIOR_SPEND_ILS before proceeding")
drive.mount("/content/drive")
OUTPUT_ROOT = Path("/content/drive/MyDrive/NLP/evaluation-runs") / RUN_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
probe = OUTPUT_ROOT / f"drive-access-{time.time_ns()}.txt"
with probe.open("x") as stream:
    stream.write(RUN_NAME)
if probe.read_text() != RUN_NAME:
    raise RuntimeError("Drive read-after-write check failed")
AVAILABILITY = {name: {"status": "not_selected"} for name in SYSTEM_NAMES}
SECRET_NAMES = {"gemini": "GEMINI_API_KEY", "openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY", "xai": "XAI_API_KEY"}
for name in SELECTED_SYSTEMS:
    if name not in SECRET_NAMES:
        continue
    secret_name = SECRET_NAMES[name]
    os.environ.pop(secret_name, None)  # Remove stale credentials before reading this notebook's Secrets.
    try:
        secret_value = userdata.get(secret_name)
        if not secret_value:
            raise ValueError("empty")
        os.environ[secret_name] = secret_value
        AVAILABILITY[name] = {"status": "credentials_loaded"}
        del secret_value
    except Exception:
        AVAILABILITY[name] = {"status": "unavailable", "reason": "missing_secret"}
ENVIRONMENT = {
    "code_revision": CODE_REVISION, "python": platform.python_version(),
    "packages": {name: importlib.metadata.version(name) for name in
                 ("torch", "transformers", "requests", "numpy")},
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "archive_sha256": ARCHIVE_SHA256 if SOURCE_MODE == "archive" else None,
}
with (OUTPUT_ROOT / f"environment-{time.time_ns()}.json").open("x") as stream:
    json.dump(ENVIRONMENT, stream, indent=2)
print(ENVIRONMENT)
print("Persistent outputs:", OUTPUT_ROOT)
print(AVAILABILITY)

Mounted at /content/drive
{'code_revision': '64f47b76317d61868765e4ad05ecf12ddc6f4379', 'python': '3.13.15', 'packages': {'torch': '2.8.0', 'transformers': '4.57.6', 'requests': '2.32.5', 'numpy': '2.5.3'}, 'gpu': 'Tesla T4', 'archive_sha256': '7f4b426451e1c38777b3f169db76b4dd56bc65aae4f67d391e72914414dbf7d2'}
Persistent outputs: /content/drive/MyDrive/NLP/evaluation-runs/evaluation-xai-qwen14-20261009-64f47b7
{'qwen': {'status': 'not_selected'}, 'gemini': {'status': 'not_selected'}, 'openai': {'status': 'not_selected'}, 'anthropic': {'status': 'not_selected'}, 'qwen14': {'status': 'not_selected'}, 'xai': {'status': 'credentials_loaded'}}


## 3. Local Model Configuration

Selecting `qwen` downloads **Qwen2.5 7B** (`qwen2.5:7b`); selecting `qwen14` downloads **Qwen2.5 14B** (`qwen2.5:14b`, Q4_K_M). Ollama is installed only when either is selected. Use a free GPU when available. Before the pilot, inspect the resolved model digest, server version, template, quantization, stop sequences, and generation settings. These properties identify the local model used in the run; the tag alone is insufficient.

Generation uses temperature 0, seed 42, and a 512-token output limit. Hardware and implementation differences may still affect reproducibility. A setup failure is recorded without substituting another model, and other available systems may proceed.


In [4]:
QWEN_MODEL = "qwen2.5:7b"
QWEN14_MODEL = "qwen2.5:14b"
QWEN_OPTIONS = {"temperature": 0, "seed": 42, "num_predict": 512}
OLLAMA_URL = "http://localhost:11434"
QWEN_IDENTITIES = {}
QWEN_STABLE_FIELDS = ("model", "digest", "server_version", "options",
                      "template", "model_parameters", "details", "model_info")
import requests
for qwen_name, qwen_model in (("qwen", QWEN_MODEL), ("qwen14", QWEN14_MODEL)):
    if qwen_name not in SELECTED_SYSTEMS:
        continue
    try:
        if shutil.which("ollama") is None:
            subprocess.run(["apt-get", "-qq", "update"], check=True)
            subprocess.run(["apt-get", "-qq", "install", "-y", "zstd"], check=True)
            installer = Path("/content/ollama-install.sh")
            urllib.request.urlretrieve("https://ollama.com/install.sh", installer)
            subprocess.run(["sh", str(installer)], check=True)
        import requests
        try:
            requests.get(OLLAMA_URL + "/api/version", timeout=5).raise_for_status()
        except requests.RequestException:
            ollama_log = (OUTPUT_ROOT / f"ollama-{time.time_ns()}.log").open("w")
            ollama_process = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log)
            for _ in range(30):
                time.sleep(1)
                try:
                    requests.get(OLLAMA_URL + "/api/version", timeout=2).raise_for_status()
                    break
                except requests.RequestException:
                    pass
            else:
                raise RuntimeError("Ollama did not start; inspect its saved log")
        subprocess.run(["ollama", "pull", qwen_model], check=True)
        inspection = inspect_ollama(qwen_model, options=QWEN_OPTIONS)
        if qwen_name == "qwen14" and inspection.get("details", {}).get("quantization_level") != "Q4_K_M":
            raise ValueError("Qwen14 quantization differs from the specified Q4_K_M")
        QWEN_IDENTITIES[qwen_name] = inspection
        # Download timestamps are session evidence, separate from stable model identity.
        with (OUTPUT_ROOT / f"{qwen_name}-inspection-{time.time_ns()}.json").open("x") as stream:
            json.dump(inspection, stream, ensure_ascii=False, indent=2)
        frozen_identity = {key: inspection[key] for key in QWEN_STABLE_FIELDS}
        AVAILABILITY[qwen_name] = {"status": "available"}
        print(json.dumps(frozen_identity, ensure_ascii=False, indent=2))
    except Exception:
        AVAILABILITY[qwen_name] = {"status": "unavailable", "reason": "local_model_setup_failed"}
print({name: AVAILABILITY[name] for name in ("qwen", "qwen14")})

{
  "model": "qwen2.5:14b",
  "digest": "7cdf5a0187d5c58cc5d369b255592f7841d1c4696d45a8c8a9489440385b22f6",
  "server_version": "0.40.2",
  "options": {
    "temperature": 0,
    "seed": 42,
    "num_predict": 512
  },
  "template": "{{- if .Messages }}\n{{- if or .System .Tools }}<|im_start|>system\n{{- if .System }}\n{{ .System }}\n{{- end }}\n{{- if .Tools }}\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>\n{{- range .Tools }}\n{\"type\": \"function\", \"function\": {{ .Function }}}\n{{- end }}\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call>\n{{- end }}<|im_end|>\n{{ end }}\n{{- range $i, $_ := .Messages }}\n{{- $last := eq (len (slice $.Messages $i)) 1 -}}\n{{- if eq .Role \"user\" }}<|im_star

## 4. API Model Availability and Cost Assumptions

The API systems are **Gemini** (`gemini-3.8-flash`), **GPT-4.1 mini** (`gpt-4.1-mini-2025-04-14`), and **Claude Haiku 5.5** (`claude-haiku-5-5`). Availability lookups do not generate text and do not establish sufficient credit or quota. The report distinguishes unavailable and unselected systems; only available selected systems enter collection.

Gemini uses low thinking, and Claude uses adaptive thinking with low effort. Output limits include provider-billed thinking where applicable. Exact settings are recorded with the run, and returned model identities are inspected in the pilot.

The rates below are expressed in USD per million tokens. Verify them against the official [Gemini pricing](https://ai.google.dev/gemini-api/docs/pricing), [GPT-4.1 mini specification](https://developers.openai.com/api/docs/models/gpt-4.1-mini), and [Claude Haiku 5.5 specification](https://platform.claude.com/docs/en/models/haiku-5-5/overview) before collection. Cached input is conservatively charged at the full input rate. Token-based expenditure is an estimate from reported usage, not a settled provider invoice.

Grok 4.7 uses low reasoning effort, no tools, and a 1,024-token output cap.
Its input/output rates are USD 2/6 per million tokens; cached input is conservatively charged at the full input rate.
Qwen2.5 14B uses the same sampling settings as 7B; its resolved digest and quantization are recorded separately.
These additions require their own inspected technical pilots. Earlier collection packages and results remain separate.
Sources: [xAI model specification](https://docs.x.ai/developers/models/grok-4.7),
[Qwen2.5 model library](https://ollama.com/library/qwen2.5:14b).


In [5]:
SYSTEMS = [
    {"name": "qwen", "provider": "qwen", "model": QWEN_MODEL,
     "settings": {"expected_digest": QWEN_IDENTITIES.get("qwen", {}).get("digest"), "base_url": OLLAMA_URL,
                  "timeout": 120, "options": QWEN_OPTIONS}},
    {"name": "gemini", "provider": "gemini", "model": "gemini-3.8-flash",
     "settings": {"timeout": 120, "generation_config": {
         "maxOutputTokens": 1024,
         "thinkingConfig": {"thinkingLevel": "low"}}}},
    {"name": "openai", "provider": "openai", "model": "gpt-4.1-mini-2025-04-14",
     "settings": {"timeout": 120, "max_output_tokens": 512, "temperature": 0}},
    {"name": "anthropic", "provider": "anthropic", "model": "claude-haiku-5-5",
     "settings": {"timeout": 120, "max_output_tokens": 1024, "effort": "low"}},
    {"name": "qwen14", "provider": "qwen", "model": QWEN14_MODEL,
     "settings": {"expected_digest": QWEN_IDENTITIES.get("qwen14", {}).get("digest"), "base_url": OLLAMA_URL,
                  "timeout": 120, "options": QWEN_OPTIONS}},
    {"name": "xai", "provider": "xai", "model": "grok-4.7",
     "settings": {"timeout": 120, "max_output_tokens": 1024, "effort": "low"}},
]
RATES = {"qwen": {"input": 0.0, "output": 0.0},
         "gemini": {"input": 0.75, "output": 3.75},
         "openai": {"input": 0.40, "output": 1.60},
         "anthropic": {"input": 0.10, "output": 0.50},
         "qwen14": {"input": 0.0, "output": 0.0},
         "xai": {"input": 2.0, "output": 6.0}}
for system in SYSTEMS:
    name = system["name"]
    if name not in SELECTED_SYSTEMS or name in {"qwen", "qwen14"} or AVAILABILITY[name]["status"] == "unavailable":
        continue
    if name == "openai":
        info = inspect_openai_model(model=system["model"])
    elif name == "anthropic":
        info = inspect_anthropic_model(model=system["model"])
    elif name == "xai":
        info = inspect_xai_model(model=system["model"])
    else:
        payload, info = request_json(
            requests.get, "https://generativelanguage.googleapis.com/v1beta/models/" + system["model"],
            provider="Gemini", secret=os.environ["GEMINI_API_KEY"], env_name="GEMINI_API_KEY",
            headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]}, timeout=30)
        if payload is not None and payload.get("name") == "models/" + system["model"] and "generateContent" in (payload.get("supportedGenerationMethods") or []):
            info["status"] = "available"
    # Save only allowlisted diagnostics, never upstream body/message or credentials.
    AVAILABILITY[name] = {key: info.get(key) for key in
        ("status", "http_status", "error_category", "provider_blocked", "retry_after")}
def available_selected_systems():
    # Recheck current selection and availability before collection.
    if not SELECTED_SYSTEMS or len(set(SELECTED_SYSTEMS)) != len(SELECTED_SYSTEMS) or not set(SELECTED_SYSTEMS) <= set(SYSTEM_NAMES):
        raise ValueError("Select approved systems without duplicates")
    return [s for s in SYSTEMS if s["name"] in SELECTED_SYSTEMS
            and AVAILABILITY.get(s["name"], {}).get("status") == "available"]
with (OUTPUT_ROOT / f"availability-{time.time_ns()}.json").open("x") as stream:
    json.dump(AVAILABILITY, stream, indent=2)
print(json.dumps(AVAILABILITY, indent=2))
DEV_PATH = REPO_DIR / "data/splits/dev_items.csv"
TEST_PATH = REPO_DIR / "data/splits/test_items.csv"
def model_identity(system):
    return ({key: QWEN_IDENTITIES[system["name"]][key] for key in QWEN_STABLE_FIELDS}
            if system["provider"] == "qwen" else None)

{
  "qwen": {
    "status": "not_selected"
  },
  "gemini": {
    "status": "not_selected"
  },
  "openai": {
    "status": "not_selected"
  },
  "anthropic": {
    "status": "not_selected"
  },
  "qwen14": {
    "status": "available"
  },
  "xai": {
    "status": "available",
    "http_status": 200,
    "error_category": null,
    "provider_blocked": false,
    "retry_after": null
  }
}


## 5. Technical Development Pilot

Each system receives the same first ten development items in both tasks: **20 logical responses per system**. Candidate order is derived from the item identity and a fixed candidate-order seed of 42. Each request is journalled before execution and its outcome after return.

At most two attempts are allowed per logical request. Explicit provider blocks stop collection for that provider, and transient retries obey bounded waiting. Interrupted requests with uncertain completion are retained for inspection and are not automatically resent. Their possible expenditure remains included through conservative allowances.

Pilot and test journals for every system contribute to the shared expenditure ceiling, including systems not currently selected. Preserve all journals when resuming. An incomplete pilot does not qualify its system for test evaluation.


In [6]:
# Upper input allowance: prompt UTF-8 bytes plus provider-envelope margin.
from hebrew_acronyms.models.common.pairs import load_rows
from hebrew_acronyms.models.common.eval import build_generate_prompt, build_select_prompt
all_rows = load_rows(DEV_PATH) + load_rows(TEST_PATH)
max_prompt_bytes = max(len(prompt.encode("utf-8")) for row in all_rows for prompt in (
    build_generate_prompt(row["acronym"], row["sentence"]),
    build_select_prompt(row["acronym"], row["sentence"], row["candidates"].split("|")),
))
input_token_allowance = max_prompt_bytes + 1024
if input_token_allowance >= 100_000:
    raise ValueError("Recheck long-context price tier before any calls")
OUTPUT_CAPS = {"qwen": 512, "gemini": 1024, "openai": 512, "anthropic": 1024, "qwen14": 512, "xai": 1024}
RESERVE_PER_CALL_USD = {
    name: (input_token_allowance * rate["input"] + OUTPUT_CAPS[name] * rate["output"]) / 1_000_000
    for name, rate in RATES.items()
}
def current_session():
    return prepare_session(
        OUTPUT_ROOT, DEV_PATH, TEST_PATH, code_revision=CODE_REVISION,
        prior_spend_ils=PRIOR_SPEND_ILS, rates=RATES, reserves=RESERVE_PER_CALL_USD,
        ils_per_usd=ILS_PER_USD_ALLOWANCE,
    )
current_session()
for system in available_selected_systems():
    result = run_system_pilot(OUTPUT_ROOT, system, model_identity=model_identity(system))
    print(system["name"], result["manifest"]["identity_sha256"],
          {key: result["summary"][key] for key in ("n_calls", "n_completed", "n_pending", "n_ambiguous")})
print(json.dumps(session_summary(OUTPUT_ROOT), indent=2))

qwen14 fc5fb8eb01aeb9e1bdbc372c722344e6d37a2a1c48dc6190bca917dcfe063299 {'n_calls': 20, 'n_completed': 19, 'n_pending': 0, 'n_ambiguous': 0}
xai 85d1efa21116c83b170c706b9d4a7bc01fe7638553534936c99970b5e4704885 {'n_calls': 20, 'n_completed': 0, 'n_pending': 0, 'n_ambiguous': 0}
{
  "systems": {
    "qwen": {
      "dev_pilot": {
        "status": "not_run",
        "n_calls": 0,
        "reason": "No saved pilot"
      },
      "full_test": {
        "status": "not_run",
        "n_calls": 0,
        "reason": "No eligible inspected pilot/full run"
      }
    },
    "gemini": {
      "dev_pilot": {
        "status": "not_run",
        "n_calls": 0,
        "reason": "No saved pilot"
      },
      "full_test": {
        "status": "not_run",
        "n_calls": 0,
        "reason": "No eligible inspected pilot/full run"
      }
    },
    "openai": {
      "dev_pilot": {
        "status": "not_run",
        "n_calls": 0,
        "reason": "No saved pilot"
      },
      "full_test": {
  

## 6. Pilot Inspection and Test Projection

Inspect all 20 responses for coverage, model identity, finish reason, truncation, token usage, elapsed time, and estimated cost. Pilot acceptance is technical: semantic errors remain observations and do not justify repeating requests or selecting a more favorable run.

For a complete pilot, the cost projection scales from 10 to 395 items with a 25% margin and includes accumulated expenditure across systems and allowances for uncertain usage. The elapsed-time projection is descriptive rather than a completion guarantee.

After inspecting a successful pilot's records, enter its printed `pilot_identity` in `INSPECTED_PILOTS` in the next cell. The initially empty mapping records no completed inspection. Incomplete or blocked systems remain reported, while independently qualified systems may proceed. Execute these stages sequentially; **Run all** is not a substitute for pilot inspection.


In [7]:
current_session()  # Validate the saved session against the current configuration.
for system in available_selected_systems():
    name = system["name"]
    pilot_summary = summarize_evaluation(OUTPUT_ROOT / name / "dev-pilot")
    print(name, json.dumps(pilot_summary["by_system_task"], ensure_ascii=False))
    for record in pilot_summary["records"]:
        print(json.dumps(record, ensure_ascii=False))
    try:
        review = review_system_pilot(OUTPUT_ROOT, system, model_identity=model_identity(system))
        print(name, "pilot_identity:", review["pilot_identity"])
        print(json.dumps(review["cost"], indent=2))
        print("Projected full minutes:", round(pilot_summary["timings_seconds"][name] * 395 / 10 / 60, 1))
        print("Projected total ILS including all saved systems:", review["projected_total_ils"])
    except ValueError:
        print(name, "Full test gated: pilot incomplete or identity/configuration mismatch; inspect saved evidence.")
print(json.dumps(session_summary(OUTPUT_ROOT), indent=2))

qwen14 [{"system": "qwen14", "task": "generate", "n_items": 10, "n_completed": 9, "status_counts": {"service_error": 1, "response_received": 9}, "accuracy": 0.2, "score": "historical_quote_normalized_gold_substring", "n_valid": 2}, {"system": "qwen14", "task": "select", "n_items": 10, "n_completed": 10, "status_counts": {"response_received": 10}, "accuracy": 0.9, "score": "strict_candidate_label_v1", "n_valid": 10}]
{"item_id": "ha-01004", "key": "f14e9e465db9d872dcc8799da39e31424d8c37ca9d84ed4270bc5f214ed92a47", "prompt": "בהתחשב במשפט הבא בעברית, מהו הפירוש של ראשי התיבות \"הרה״ר\"?\nענה במילים ספורות בלבד, ללא הסבר.\n\nמשפט: בראש ההנהגה הרוחנית של העדה והיישוב היהודי בארץ ישראל עמד הרה״ר, שכונה חכם באשי ובהמשך \"הראשון לציון\".\nפירוש:", "prompt_sha256": "37ffdac78467c16f3397daf36ee290ded3ba97ee03df24c0a614067626bfd040", "shown_order": null, "system": "qwen14", "task": "generate", "status": "service_error", "response": "", "selected_candidate": null, "correct": false, "valid": false

## 7. Test Evaluation and Resumption

Each qualified system is evaluated on all **395 test items in both tasks**, yielding **790 logical response records**. `INSPECTED_PILOTS` maps each inspected system name to the exact SHA-256 identity printed for its successful pilot. Before collection, the runner reloads that pilot, validates source, inputs, settings, and model identity, checks coverage, and recomputes the projection and shared expenditure.

Each additional system requires its own matching pilot. Unselected, uninspected, or blocked systems are reported explicitly. Resumption retains completed responses, failed attempts, ambiguous requests, and accumulated expenditure; completed responses are not requested again. All test identities remain in the cohort, including failed and unrun cases.

Freeze the source revision and settings during collection. A configuration change or repair of an explicit account block requires a new identified session and matching pilots, with previous artifacts preserved and expenditure carried forward. Deleting journals or changing identifiers to bypass a block is incompatible with this protocol.


In [ ]:
INSPECTED_PILOTS = {}  # Map inspected system names to their exact pilot identity SHA-256.
current_session()  # Validate the saved session against the current configuration.
for system in available_selected_systems():
    name = system["name"]
    if name not in INSPECTED_PILOTS:
        print(name, "Full test not run: pilot not inspected")
        continue
    try:
        test_summary = run_system_full(
            OUTPUT_ROOT, system, inspected_pilot_identity=INSPECTED_PILOTS[name],
            model_identity=model_identity(system),
        )
        print(name, {key: test_summary[key] for key in
              ("n_items", "n_records", "n_calls", "n_completed", "n_pending", "n_ambiguous")})
    except ValueError:
        print(name, "Full test gated: verify saved pilot identity, coverage, inputs and shared budget.")
print(json.dumps(session_summary(OUTPUT_ROOT), indent=2))

## 8. Baselines and Encoder Limitations

The deterministic baseline evaluation includes uniform-random expectation, most-frequent expansion, most-mined expansion, and oracle selection. The trained DictaBERT candidate scorer and the untrained DictaBERT similarity baseline are separate systems; this notebook does not retrain either model.

The checkpoint adapter in `src/hebrew_acronyms/models/dictabert_cross_encoder/colab.py` supports the identified weights with SHA-256 `23bbff0324a7ce4d9d4a24a9ebfb87a4f6ac296bf1f1ce1bd25126c09260267f`. Checkpoint reuse also requires the corresponding model and tokenizer snapshot. A state dictionary alone does not establish the original training seed, selected epoch, library versions, or exact training examples. See `docs/checkpoints.md` in the source archive for the inference procedure and provenance limitations.

**Encoder evaluation limitation.** The historical test CSV lacks the explicit target spans required by the strict encoder interface. Multiple acronym occurrences cannot be resolved by silently selecting the first match. Fresh encoder evaluation requires qualified spans and is not performed by this notebook. Historical encoder results remain separately identified.


In [ ]:
from hebrew_acronyms.test_baselines import run_test_baselines
baseline_summary = run_test_baselines(
    TEST_PATH, REPO_DIR / "data/mined/candidate_table.csv",
    OUTPUT_ROOT / "baselines", code_revision=CODE_REVISION,
)
print(baseline_summary["summary"])

## 9. Result Artifacts and Interpretation

Each system's pilot and test directories contain a `manifest.json` identifying code, inputs, complete prompts, candidate orders, and settings; an `attempts.jsonl` journal retaining requests and outcomes; and a regenerated `summary.json` containing coverage and response records. `session.json` records the shared session configuration. Preserve the complete Drive directory and download an archive as a second copy.

Report results by system and task with the full cohort denominator, including missing or incomplete responses. Automatic selection-label parsing, string-based scores, and output-quality flags are distinct from human semantic judgments of generated expansions. Interpretation must retain source groups, returned model versions, run identities, and judgment provenance. Candidate labels beyond Z retain items with 30 candidates; comparisons with historical runs must account for this protocol difference.

`notebooks/experimental_study.ipynb` is the methods and analysis companion in the source archive. The existing development-set paper exporter is not a full-test exporter. Only explicitly identified run artifacts and evaluation procedures support reported results; downloading an archive does not itself validate the experiment or publish its findings.


In [8]:
from google.colab import files
print(json.dumps(session_summary(OUTPUT_ROOT), ensure_ascii=False, indent=2))
for name in SYSTEM_NAMES:
    directory = OUTPUT_ROOT / name / "full-test"
    if (directory / "manifest.json").exists():
        saved_summary = summarize_evaluation(directory)
        print(json.dumps(saved_summary["by_system_task"], ensure_ascii=False, indent=2))
archive_path = shutil.make_archive("/content/" + RUN_NAME + "-" + str(time.time_ns()), "zip", OUTPUT_ROOT)
files.download(archive_path)

{
  "systems": {
    "qwen": {
      "dev_pilot": {
        "status": "not_run",
        "n_calls": 0,
        "reason": "No saved pilot"
      },
      "full_test": {
        "status": "not_run",
        "n_calls": 0,
        "reason": "No eligible inspected pilot/full run"
      }
    },
    "gemini": {
      "dev_pilot": {
        "status": "not_run",
        "n_calls": 0,
        "reason": "No saved pilot"
      },
      "full_test": {
        "status": "not_run",
        "n_calls": 0,
        "reason": "No eligible inspected pilot/full run"
      }
    },
    "openai": {
      "dev_pilot": {
        "status": "not_run",
        "n_calls": 0,
        "reason": "No saved pilot"
      },
      "full_test": {
        "status": "not_run",
        "n_calls": 0,
        "reason": "No eligible inspected pilot/full run"
      }
    },
    "anthropic": {
      "dev_pilot": {
        "status": "not_run",
        "n_calls": 0,
        "reason": "No saved pilot"
      },
      "full_test": {
 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
# One identified diagnostic request; saved pilots remain unchanged.
from unittest.mock import patch
from hebrew_acronyms.models.xai import eval as xai_adapter
from hebrew_acronyms.models.gemini.eval import _redact
_diag_root = OUTPUT_ROOT.parent.parent / "evaluation-diagnostics" / "xai-http400-20261009-64f47b7"
_diag_root.mkdir(parents=True, exist_ok=True)
_diag_path = _diag_root / "diagnostic.jsonl"
_diag_prompt = json.loads((OUTPUT_ROOT / "xai/dev-pilot/manifest.json").read_text())["identity"]["rows"][0]
_diag_prompt = build_generate_prompt(_diag_prompt["acronym"], _diag_prompt["sentence"])
_diag_prior = session_summary(OUTPUT_ROOT)["total_accounted_ils"]
_diag_reserve = RESERVE_PER_CALL_USD["xai"] * ILS_PER_USD_ALLOWANCE
assert _diag_prior + _diag_reserve <= 100
_diag_error = {}
_diag_post = requests.post
def _capture_diagnostic(*args, **kwargs):
    response = _diag_post(*args, **kwargs)
    if response.status_code >= 400:
        try:
            body = response.json()
            error = body.get("error", body)
            if isinstance(error, dict):
                _diag_error.update({k: error.get(k) for k in ("code", "type", "param", "message") if isinstance(error.get(k), (str, int))})
            elif isinstance(error, str):
                _diag_error["message"] = error
        except ValueError:
            _diag_error["message"] = "Non-JSON error omitted"
    return response
with _diag_path.open("x") as _diag_stream:
    _diag_start = {"event": "start", "source_revision": CODE_REVISION, "source_session": RUN_NAME, "prompt_sha256": hashlib.sha256(_diag_prompt.encode()).hexdigest(), "prior_accounted_ils": _diag_prior, "reserved_ils": _diag_reserve, "max_requests": 1}
    _diag_stream.write(json.dumps(_diag_start) + "\n")
    _diag_stream.flush(); os.fsync(_diag_stream.fileno())
    with patch.object(xai_adapter.requests, "post", side_effect=_capture_diagnostic):
        _diag_result = xai_adapter.xai_response(_diag_prompt, model="grok-4.7", max_output_tokens=1024, timeout=120, effort="low")
    _diag_safe_error = _redact(_diag_error, os.environ["XAI_API_KEY"])
  # One identified diagnostic request; saved pilots remain unchanged.
from unittest.mock import patch
from hebrew_acronyms.models.xai import eval as xai_adapter
from hebrew_acronyms.models.gemini.eval import _redact
_diag_root = OUTPUT_ROOT.parent.parent / "evaluation-diagnostics" / "xai-http400-20261009-64f47b7"
_diag_root.mkdir(parents=True, exist_ok=True)
_diag_path = _diag_root / "diagnostic.jsonl"
_diag_prompt = json.loads((OUTPUT_ROOT / "xai/dev-pilot/manifest.json").read_text())["identity"]["rows"][0]
_diag_prompt = build_generate_prompt(_diag_prompt["acronym"], _diag_prompt["sentence"])
_diag_prior = session_summary(OUTPUT_ROOT)["total_accounted_ils"]
_diag_reserve = RESERVE_PER_CALL_USD["xai"] * ILS_PER_USD_ALLOWANCE
assert _diag_prior + _diag_reserve <= 100
_diag_error = {}
_diag_post = requests.post
def _capture_diagnostic(*args, **kwargs):
    response = _diag_post(*args, **kwargs)
    if response.status_code >= 400:
        try:
            body = response.json()
            error = body.get("error", body)
            if isinstance(error, dict):
                _diag_error.update({k: error.get(k) for k in ("code", "type", "param", "message") if isinstance(error.get(k), (str, int))})
            elif isinstance(error, str):
                _diag_error["message"] = error
        except ValueError:
            _diag_error["message"] = "Non-JSON error omitted"
    return response
with _diag_path.open("x") as _diag_stream:
    _diag_start = {"event": "start", "source_revision": CODE_REVISION, "source_session": RUN_NAME, "prompt_sha256": hashlib.sha256(_diag_prompt.encode()).hexdigest(), "prior_accounted_ils": _diag_prior, "reserved_ils": _diag_reserve, "max_requests": 1}
    _diag_stream.write(json.dumps(_diag_start) + "\n")
    _diag_stream.flush(); os.fsync(_diag_stream.fileno())
    with patch.object(xai_adapter.requests, "post", side_effect=_capture_diagnostic):
        _diag_result = xai_adapter.xai_response(_diag_prompt, model="grok-4.7", max_output_tokens=1024, timeout=120, effort="low")
    _diag_safe_error = _redact(_diag_error, os.environ["XAI_API_KEY"])
    _diag_end = {"event": "finish", "result": _diag_result, "safe_error": _diag_safe_error, "accounted_ils_including_reserve": _diag_prior + _diag_reserve}
    _diag_stream.write(json.dumps(_diag_end, ensure_ascii=False) + "\n")
    _diag_stream.flush(); os.fsync(_diag_stream.fileno())
print({"status": _diag_result["status"], "http_status": _diag_result.get("http_status"), "safe_error": _diag_safe_error, "accounted_ils_including_reserve": _diag_prior + _diag_reserve, "evidence": str(_diag_path)})


FileExistsError: [Errno 17] File exists: '/content/drive/MyDrive/NLP/evaluation-diagnostics/xai-http400-20261009-64f47b7/diagnostic.jsonl'

In [12]:
_diag_recovered = {"event": "finish_recovered_from_kernel", "result": _diag_result, "safe_error": _diag_safe_error, "accounted_ils_including_reserve": _diag_prior + _diag_reserve}; _diag_recovery_file = (_diag_root / "recovered-finish.json").open("x"); _diag_recovery_file.write(json.dumps(_diag_recovered, ensure_ascii=False)); _diag_recovery_file.flush(); os.fsync(_diag_recovery_file.fileno()); _diag_recovery_file.close(); print("Diagnostic result preserved; no new request.")

Diagnostic result preserved; no new request.


In [13]:
files.download(shutil.make_archive("/content/xai-http400-20261009-64f47b7", "zip", _diag_root))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>